# 01 — Loan Default Prediction: Problem Definition & Comprehensive EDA
## Computer Engineering — Semester Machine Learning Project (Weeks 1 & 2)
### Objective:
Predict borrower default risk on the Kaggle Loan Default Dataset (255,347 records, 18 attributes).

### Domain Context:
In commercial and retail banking, determining credit default probability is vital for capital preservation, interest rate pricing, and regulatory risk capital reserve allocation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configure styles
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries imported successfully.")


## 1. Load Dataset & Inspect Basic Properties


In [ ]:
data_path = os.path.join("..", "Backend", "data", "Loan_default.csv")
if not os.path.exists(data_path):
    data_path = "Backend/data/Loan_default.csv"

df = pd.read_csv(data_path)
print("Dataset Shape:", df.shape)
df.head()


In [ ]:
print("Column Names and Data Types:")
print(df.dtypes)
print("\nMissing Values Check:")
print(df.isnull().sum())
print("\nDuplicate Records Check:", df.duplicated().sum())


## 2. Target Variable Analysis (Class Imbalance)
Loan default status is stored in the `Default` column (0 = Non-Default, 1 = Default).


In [ ]:
counts = df['Default'].value_counts()
proportions = df['Default'].value_counts(normalize=True) * 100

print(f"Non-Default (0): {counts[0]:,} ({proportions[0]:.2f}%)")
print(f"Default (1):     {counts[1]:,} ({proportions[1]:.2f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=['Non-Default (0)', 'Default (1)'], y=counts.values, palette=['#10b981', '#ef4444'], ax=ax)
ax.set_title("Target Class Distribution (Severe Imbalance: ~88% vs 12%)", fontweight='bold')
ax.set_ylabel("Count")
plt.show()


## 3. Numerical Features Distribution & Outliers (IQR Method)


In [ ]:
numerical_cols = ["Age", "Income", "LoanAmount", "CreditScore", "MonthsEmployed", "NumCreditLines", "InterestRate", "LoanTerm", "DTIRatio"]
df[numerical_cols].describe().T


In [ ]:
# Outlier analysis using IQR
for col in numerical_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    outliers = df[(df[col] < low) | (df[col] > high)]
    print(f"{col:<16} | Outliers: {len(outliers):>6} ({len(outliers)/len(df)*100:>5.2f}%) | Bounds: [{low:.1f}, {high:.1f}]")


## 4. Correlation Analysis


In [ ]:
plt.figure(figsize=(10, 8))
corr = df[numerical_cols + ['Default']].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", vmin=-0.2, vmax=0.2, square=True)
plt.title("Correlation Heatmap with Target Variable", fontweight='bold')
plt.show()


## 5. Categorical Feature Default Propensities


In [ ]:
categorical_cols = ["Education", "EmploymentType", "MaritalStatus", "HasMortgage", "HasDependents", "LoanPurpose", "HasCoSigner"]
for cat in categorical_cols:
    rates = df.groupby(cat)['Default'].mean() * 100
    print(f"--- Default Rate by {cat} ---")
    print(rates.round(2).to_string())
    print()
